# Notebook 06: GRMHD Bridge Demo

**Gap 6**: The paper uses a simplified broken-disk geometry. This notebook
demonstrates the pipeline that connects full GRMHD simulation outputs
(e.g. from HAMR, Liska et al. 2019) to time-resolved polarization predictions.


In [ ]:
import sys; sys.path.insert(0, '..')
import numpy as np
import h5py
import matplotlib.pyplot as plt
from src.grmhd_bridge.grmhd_to_polarization import MockGRMHDReader, run_bridge_pipeline
from src.utils.physics import ENERGY_BINS


In [ ]:
# Inspect the mock GRMHD geometry evolution
reader = MockGRMHDReader()
data = reader.load('mock')
snapshots = reader.list_snapshots(data)

times, r_bps, betas = [], [], []
for idx in snapshots:
    geom = reader.extract_geometry(data, idx)
    times.append(geom.time)
    r_bps.append(geom.r_bp)
    betas.append(geom.beta)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(times, r_bps, 'b-o')
axes[0].set_xlabel('Simulation time [M]')
axes[0].set_ylabel('r_BP [r_g]')
axes[0].set_title('Break radius evolution')
axes[0].grid(alpha=0.3)

axes[1].plot(times, betas, 'r-o')
axes[1].set_xlabel('Simulation time [M]')
axes[1].set_ylabel('β [degrees]')
axes[1].set_title('Misalignment angle evolution')
axes[1].grid(alpha=0.3)

plt.tight_layout()
plt.savefig('../results/figures/grmhd_geometry_evolution.png', dpi=150)
plt.show()
print('This mimics the disk alignment shown in Liska et al. (2019): r_BP shrinks as disk aligns.')


In [ ]:
# Run the full pipeline (requires trained emulator)
# run_bridge_pipeline(
#     grmhd_path='mock',
#     emulator_path='../results/models/emulator_best.pt',
#     output_path='../results/reports/grmhd_polarization.h5',
#     reader_type='mock'
# )

# Show what the output looks like conceptually
print('Output structure: grmhd_polarization.h5')
print('  /time          (N_t,)              — simulation time')
print('  /geometry      (N_t, 5)            — [spin, r_bp, beta, phi, inclination]')
print('  /polarization  (N_t, N_energy, 2)  — [pol_frac, pol_angle] time-series')
print('  /energy_keV    (N_energy,)')
print()
print('This enables direct comparison with IXPE light curves during outburst events.')
